# Tune `mspc_rf`

MSPC + Random Forest. Repeated stratified CV on the train split; 
writes [`data/processed/tuned/mspc_rf.json`](../data/processed/tuned/mspc_rf.json).

**Stage 1 (hyperparameters):** PLS `n_components`, classifier `max_depth`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_ID = "mspc_rf"
spec = MODEL_SPECS[MODEL_ID]


In [2]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [3]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_mspc__pls__n_components,classifier__max_depth
0,"[15, 20, 25, 30]",NaN
1,NaN,"[3, 4, 6, 8]"


In [4]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


mspc_rf: 16 candidates x 25 folds = 400 fits


GridSearchCV 400 fits:   0%|          | 0/400 [00:00<?, ?it/s]

  0%|          | 0/400 [00:00<?, ?it/s]

Fitting 25 folds for each of 16 candidates, totalling 400 fits


In [5]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,n_components,max_depth,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
2,15,6,42.978695,5.267455,0.570213,19.632353,10.418368,94.410256,1.424661,0.706160,0.071508,0.189140,0.068210
3,15,8,43.628896,4.792723,0.563711,17.955882,9.386847,94.786325,1.425071,0.704027,0.071724,0.187607,0.064369
5,20,4,42.595840,3.956937,0.574042,21.338235,7.890963,93.470085,1.407119,0.710831,0.065069,0.187273,0.061528
7,20,8,43.856146,3.974578,0.561439,16.544118,7.901590,95.743590,1.181096,0.709089,0.068454,0.186783,0.063068
4,20,3,41.121858,4.615736,0.588781,26.132353,9.391016,91.623932,1.598999,0.709072,0.064534,0.186759,0.060822
1,15,4,41.236425,5.403788,0.587636,24.911765,10.802709,92.615385,1.573764,0.707134,0.067843,0.186698,0.061603
6,20,6,43.560080,3.779832,0.564399,17.529412,7.302563,95.350427,1.298020,0.711336,0.066982,0.186323,0.061602
0,15,3,40.961350,5.079345,0.590387,27.102941,10.569080,90.974359,1.663135,0.703737,0.067040,0.185191,0.063022
13,30,4,43.562783,3.841117,0.564372,18.720588,7.889894,94.153846,1.418494,0.706308,0.067053,0.182738,0.057763
14,30,6,44.858912,3.735848,0.551411,14.367647,7.429688,95.914530,1.184802,0.707088,0.068935,0.182104,0.056771


In [6]:
threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
print(f"Stage 2 best threshold: {threshold_result['best_threshold']:.2f}")
print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/25 [00:00<?, ?it/s]

Stage 2 best threshold: 0.25
  mean BER at threshold: 32.84%


,threshold,mean_ber_percent
0,0.25,32.839367
1,0.20,33.979638
2,0.30,34.939480
3,0.15,35.445513
4,0.35,36.758358
5,0.40,39.066428
6,0.10,39.870287
7,0.45,40.689165
8,0.50,42.978695
9,0.55,44.672888


In [7]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/mspc_rf.json


{'preprocess__sensor_mspc__pls__n_components': 15, 'classifier__max_depth': 6}